# Build an Evidence-Grounded AML Investigation Agent on Amazon Bedrock

This notebook builds a bounded anti-money-laundering (AML) investigation assistant with the OpenAI Agents SDK and an OpenAI model on Amazon Bedrock. The example combines:

- deterministic application tools for retrieving evidence and detecting transparent signals;
- an agent that decides how to use those tools and synthesizes a typed assessment;
- evidence-ID validation and small behavioral evals; and
- a clear human-review boundary.

All names, accounts, and transactions in this notebook are synthetic. This is an educational engineering pattern—not legal advice, a transaction-monitoring system, a filing decision, or a production compliance product.

This example complements [Getting Started with OpenAI Models on Amazon Bedrock](https://cookbook.openai.com/examples/partners/AWS/openai_models_with_amazon_bedrock): that guide surveys the Bedrock Responses API surface, while this notebook focuses narrowly on the Agents SDK tool loop, evidence grounding, typed output, and behavioral evaluation.

## What you will learn

By the end, you will be able to:

1. route the OpenAI Agents SDK through Amazon Bedrock using the AWS credential chain;
2. expose read-only evidence functions as agent tools;
3. constrain the final answer with a Pydantic output type;
4. observe the tool loop that `Runner` executes; and
5. evaluate tool use, evidence grounding, and prohibited claims.

The design starts with one specialist. Add more agents only when a distinct specialist needs different tools, instructions, or approval boundaries.

## Architecture

```text
Synthetic case ID
      |
      v
Agents SDK Runner  <---->  OpenAI model on Amazon Bedrock
      |
      +--> get_case_profile()
      +--> list_case_transactions()
      +--> run_typology_checks()
      |
      v
Typed RiskAssessment
      |
      +--> deterministic grounding and policy checks
      |
      v
Qualified human review
```

The model interprets evidence; application code owns the source data, deterministic rules, validation, and any downstream approval gate.

## 1. Install dependencies

Run the notebook in an environment with Python 3.10 or newer. Restart the kernel if the installation changes packages already imported in the session.

In [ ]:
%pip install -U "openai[bedrock]>=2.46.0" "openai-agents>=0.18.2" "pydantic>=2.13.0" --quiet

## 2. Configure Amazon Bedrock

This example uses the standard AWS credential chain. Configure AWS SSO, environment variables, a container role, or an instance role outside the notebook. Never paste access keys, secret keys, session tokens, or SSO cache contents into a notebook.

Set the Region before starting Jupyter, for example:

```bash
export AWS_PROFILE=YOUR_PROFILE
export AWS_REGION=us-east-2
jupyter lab
```

The model and Region pairing must be available in your AWS account. AWS documents GPT-5.6 Sol in US East (N. Virginia) and US East (Ohio); this notebook defaults to `us-east-2`. The `/models` preflight below checks the same Bedrock Mantle provider path without running inference.

In [ ]:
from __future__ import annotations

import json
import os
from typing import Literal

from agents import (
    Agent,
    ModelSettings,
    RunConfig,
    Runner,
    function_tool,
    set_default_openai_api,
    set_default_openai_client,
    set_tracing_disabled,
)
from agents.items import ToolCallItem
from openai import AsyncBedrockOpenAI
from openai.types.shared import Reasoning
from pydantic import BaseModel, Field

AWS_REGION = os.getenv("AWS_REGION", "us-east-2")
MODEL_ID = os.getenv("BEDROCK_MODEL", "openai.gpt-5.6-sol")

client = AsyncBedrockOpenAI(aws_region=AWS_REGION)
models_client = AsyncBedrockOpenAI(
    aws_region=AWS_REGION,
    base_url=f"https://bedrock-mantle.{AWS_REGION}.api.aws/v1",
)
set_default_openai_client(client, use_for_tracing=False)
set_default_openai_api("responses")

# Traces normally export to the OpenAI Platform. This Bedrock-only notebook
# keeps tracing local and disabled because it does not configure an OpenAI API key.
set_tracing_disabled(True)

print({"region": AWS_REGION, "model": MODEL_ID})

### 2.1 Preflight model access

Listing models is a safer first check than sending a paid inference request. Bedrock Mantle exposes model discovery at `/v1/models`, while OpenAI model inference uses `/openai/v1/responses`, so the notebook uses separate clients rooted at those two documented paths.

In [ ]:
available_models = await models_client.models.list()
available_model_ids = sorted(model.id for model in available_models.data)

if MODEL_ID not in available_model_ids:
    raise RuntimeError(
        f"{MODEL_ID!r} is not visible in {AWS_REGION}. "
        "Verify the AWS account, Region, and Bedrock model access."
    )

print(f"Preflight passed: {MODEL_ID} is visible in {AWS_REGION}.")

## 3. Create a small synthetic investigation

The case includes several cash credits below a round threshold followed by a rapid outbound transfer. Those facts are intentionally simple so we can inspect every rule and citation. They are not a regulatory definition and should not be reused as production policy.

In [ ]:
SYNTHETIC_CASE = {
    "case_id": "SYNTH-AML-001",
    "subject": "Northstar Imports LLC",
    "subject_type": "BUSINESS",
    "stated_business": "Wholesale home goods",
    "risk_tier": "STANDARD",
    "alert_reason": "Unusual cash activity followed by an outbound wire",
    "transactions": [
        {
            "id": "TXN-001",
            "timestamp": "2026-05-04T09:20:00Z",
            "direction": "CREDIT",
            "channel": "CASH",
            "amount": 9200,
            "currency": "USD",
            "counterparty": "Synthetic cash deposit A",
            "country_code": "US",
        },
        {
            "id": "TXN-002",
            "timestamp": "2026-05-04T11:05:00Z",
            "direction": "CREDIT",
            "channel": "CASH",
            "amount": 9500,
            "currency": "USD",
            "counterparty": "Synthetic cash deposit B",
            "country_code": "US",
        },
        {
            "id": "TXN-003",
            "timestamp": "2026-05-04T13:40:00Z",
            "direction": "CREDIT",
            "channel": "CASH",
            "amount": 9800,
            "currency": "USD",
            "counterparty": "Synthetic cash deposit C",
            "country_code": "US",
        },
        {
            "id": "TXN-004",
            "timestamp": "2026-05-04T16:10:00Z",
            "direction": "DEBIT",
            "channel": "WIRE",
            "amount": 28200,
            "currency": "USD",
            "counterparty": "Synthetic overseas supplier",
            "country_code": "GB",
        },
    ],
}

VALID_EVIDENCE_IDS = {
    transaction["id"] for transaction in SYNTHETIC_CASE["transactions"]
}

print(json.dumps(SYNTHETIC_CASE, indent=2))

## 4. Define the typed output contract

A structured output makes the agent result easier to validate and safer to pass into downstream application logic. It does not make the model's conclusions automatically correct; evidence and policy checks still matter.

In [ ]:
class Finding(BaseModel):
    finding_type: Literal["STRUCTURING_SIGNAL", "RAPID_MOVEMENT_SIGNAL"]
    title: str
    explanation: str
    severity: int = Field(ge=1, le=10)
    evidence_ids: list[str] = Field(min_length=1)


class EvidenceCitation(BaseModel):
    claim: str
    evidence_ids: list[str] = Field(min_length=1)


class RiskAssessment(BaseModel):
    case_id: str
    risk_level: Literal["LOW", "MEDIUM", "HIGH", "CRITICAL"]
    risk_score: int = Field(ge=0, le=100)
    executive_summary: str
    findings: list[Finding]
    information_gaps: list[str]
    recommended_next_steps: list[str]
    citations: list[EvidenceCitation] = Field(min_length=1)

## 5. Build read-only evidence tools

The first two tools retrieve source facts. The third runs transparent application-owned checks. The checks produce investigation signals—not legal conclusions or filing decisions.

In [ ]:
def require_known_case(case_id: str) -> None:
    if case_id != SYNTHETIC_CASE["case_id"]:
        raise ValueError(f"Unknown synthetic case: {case_id}")


@function_tool
def get_case_profile(case_id: str) -> str:
    """Return the synthetic customer profile and alert context for one case."""

    require_known_case(case_id)
    profile = {
        key: value
        for key, value in SYNTHETIC_CASE.items()
        if key != "transactions"
    }
    return json.dumps(profile)


@function_tool
def list_case_transactions(case_id: str) -> str:
    """Return all synthetic transactions and their evidence identifiers."""

    require_known_case(case_id)
    return json.dumps(SYNTHETIC_CASE["transactions"])


def detect_typology_signals() -> list[dict]:
    transactions = SYNTHETIC_CASE["transactions"]
    cash_credits = [
        transaction
        for transaction in transactions
        if transaction["direction"] == "CREDIT"
        and transaction["channel"] == "CASH"
        and 9000 <= transaction["amount"] < 10000
    ]
    outbound_wires = [
        transaction
        for transaction in transactions
        if transaction["direction"] == "DEBIT"
        and transaction["channel"] == "WIRE"
    ]

    findings = []
    if len(cash_credits) >= 3:
        findings.append(
            {
                "finding_type": "STRUCTURING_SIGNAL",
                "explanation": (
                    "Three same-day synthetic cash credits fall within the "
                    "demo rule's configured amount band."
                ),
                "evidence_ids": [item["id"] for item in cash_credits],
            }
        )

    credited_amount = sum(item["amount"] for item in cash_credits)
    rapid_wires = [
        item for item in outbound_wires if item["amount"] >= credited_amount * 0.9
    ]
    if cash_credits and rapid_wires:
        findings.append(
            {
                "finding_type": "RAPID_MOVEMENT_SIGNAL",
                "explanation": (
                    "A same-day synthetic outbound wire is at least 90% of the "
                    "cash credited under the demo rule."
                ),
                "evidence_ids": [
                    *[item["id"] for item in cash_credits],
                    *[item["id"] for item in rapid_wires],
                ],
            }
        )

    return findings


@function_tool
def run_typology_checks(case_id: str) -> str:
    """Run transparent demo checks and return traceable investigation signals."""

    require_known_case(case_id)
    return json.dumps(detect_typology_signals())


print(json.dumps(detect_typology_signals(), indent=2))

## 6. Define the analysis agent

The instructions require the agent to use all three tools, distinguish signals from conclusions, cite evidence IDs, identify gaps, and stop before SAR drafting or filing decisions.

In [ ]:
analysis_agent = Agent(
    name="Synthetic AML Risk Analysis Agent",
    model=MODEL_ID,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="medium"),
        store=False,
    ),
    output_type=RiskAssessment,
    tools=[
        get_case_profile,
        list_case_transactions,
        run_typology_checks,
    ],
    instructions=(
        "Analyze exactly one synthetic AML case. Call get_case_profile, "
        "list_case_transactions, and run_typology_checks before reaching a "
        "conclusion. Treat deterministic findings as investigation signals, "
        "not legal conclusions. Never invent transactions, identities, "
        "jurisdictions, or evidence IDs. Cite supplied transaction IDs for "
        "every material factual claim. Identify information gaps. Do not draft "
        "a SAR, change case state, recommend enforcement, or claim that filing "
        "is required. Return only the RiskAssessment schema."
    ),
)

## 7. Run the Agents SDK tool loop

`Runner.run` sends the task to the model, executes requested tools, returns tool results to the model, and stops when the typed final output is complete. This cell performs paid inference in your AWS account. Review the selected AWS account, Region, model, and applicable pricing before running it.

In [ ]:
result = await Runner.run(
    analysis_agent,
    f"Analyze synthetic case {SYNTHETIC_CASE['case_id']}.",
    max_turns=8,
    run_config=RunConfig(
        tracing_disabled=True,
        workflow_name="Evidence-grounded synthetic AML analysis",
    ),
)

assessment = result.final_output
tool_calls = [
    item.raw_item.name
    for item in result.new_items
    if isinstance(item, ToolCallItem)
]

print("Tool calls:", tool_calls)
print(assessment.model_dump_json(indent=2))

## 8. Evaluate the behavior

A valid JSON shape is necessary but insufficient. These checks test the behavior that matters for this workflow:

- all required evidence tools ran;
- every citation points to a real transaction in the case;
- both deterministic signals are represented;
- the output covers the evidence used by those signals; and
- the agent does not claim a SAR was filed or that filing is required.

Production evals should add more scenarios, missing-evidence cases, adversarial inputs, expert-reviewed golden cases, latency, and cost thresholds.

In [ ]:
required_tools = {
    "get_case_profile",
    "list_case_transactions",
    "run_typology_checks",
}
observed_finding_types = {
    finding.finding_type for finding in assessment.findings
}
cited_evidence_ids = {
    evidence_id
    for citation in assessment.citations
    for evidence_id in citation.evidence_ids
}
finding_evidence_ids = {
    evidence_id
    for finding in assessment.findings
    for evidence_id in finding.evidence_ids
}
serialized_assessment = assessment.model_dump_json().casefold()
forbidden_claims = [
    "sar was filed",
    "sar has been filed",
    "filing is required",
]

checks = {
    "case identity": assessment.case_id == SYNTHETIC_CASE["case_id"],
    "required tools": required_tools.issubset(set(tool_calls)),
    "expected signals": observed_finding_types
    == {"STRUCTURING_SIGNAL", "RAPID_MOVEMENT_SIGNAL"},
    "valid citations": bool(cited_evidence_ids)
    and cited_evidence_ids.issubset(VALID_EVIDENCE_IDS),
    "finding evidence covered": finding_evidence_ids.issubset(cited_evidence_ids),
    "human review boundary": not any(
        phrase in serialized_assessment for phrase in forbidden_claims
    ),
}

for check_name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check_name}")

assert all(checks.values()), "One or more behavioral checks failed."
print("All behavioral checks passed.")

## 9. Production hardening

This notebook proves an agent pattern, not a production AML system. Before using a related design with real customer data, teams should add at least:

- customer-approved typology policy and qualified compliance review;
- enterprise identity, authorization, encryption, network controls, and data classification;
- controlled persistence, immutable audit records, and retention policy;
- prompt-injection and tool-abuse defenses at every data boundary;
- human approval before any case-state change, narrative release, or filing action;
- golden-case evals reviewed by domain experts, plus regression, latency, and cost gates;
- operational monitoring, retry and idempotency behavior, and incident response; and
- independent validation of model, Region, privacy, retention, and feature requirements.

A useful next step is to place the same agent behind a private API and static web application while keeping evidence tools and approval actions server-side. A Q&A specialist and a SAR-drafting specialist should be added only after their separate permissions, tools, outputs, and eval criteria are defined.

## References

- [OpenAI models in Amazon Bedrock](https://developers.openai.com/api/docs/guides/amazon-bedrock)
- [OpenAI Agents SDK guide](https://developers.openai.com/api/docs/guides/agents)
- [OpenAI Agents SDK for Python](https://github.com/openai/openai-agents-python)
- [Evaluate agent workflows](https://developers.openai.com/api/docs/guides/agent-evals)
- [GPT-5.6 Sol model documentation](https://developers.openai.com/api/docs/models/gpt-5.6-sol)
- [Amazon Bedrock GPT-5.6 Sol model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-openai-gpt-56-sol.html)
- [AWS announcement: OpenAI GPT-5.6 models on Amazon Bedrock](https://aws.amazon.com/about-aws/whats-new/2026/07/openai-gpt-sol-terra/)